<a href="https://colab.research.google.com/github/rathodgayathri9-glitch/neural-networks-machine-learning-exercises/blob/main/Exercise-05-MADALINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Exercise 5: MADALINE-Based Heart Disease Classification and Adaptive Decision Analysis System

Dataset : Heart Disease (UCI / Kaggle "heart_disease_uci.csv")
Classes : 0 = Absence of heart disease, 1 = Presence of heart disease
Network : MADALINE = several ADALINE hidden units + fixed majority-vote output unit
Training: each ADALINE is first pre-trained with the Widrow-Hoff (LMS) rule on a bootstrap
          sample (gives the units diversity), then the whole network is refined with
          MADALINE Rule I (MRI) using the *minimal disturbance principle*.

Usage   : python exercise5_madaline_heart.py [path_to_csv]
          (looks for heart_disease_uci.csv, heart.csv, heart_disease.csv or processed.cleveland.data
           in the current folder if no path is given)
Needs   : numpy, pandas, urllib
"""
import os
import sys
import time
import urllib.request
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------------------
# Data loading and preprocessing
# ----------------------------------------------------------------------------------------
CLEVELAND_COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
                  "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]


def download_uci_dataset(filename="processed.cleveland.data"):
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
    print(f"Dataset not found locally. Downloading fallback dataset from UCI: {url} ...")
    try:
        urllib.request.urlretrieve(url, filename)
        print("Download complete.")
    except Exception as e:
        raise SystemExit(f"Failed to download dataset: {e}")


def load_heart(path=None):
    candidates = ([path] if path else []) + [
        "heart_disease_uci.csv", "heart.csv", "heart_disease.csv", "processed.cleveland.data"]
    f = next((p for p in candidates if p and os.path.exists(p)), None)
    if f is None:
        # Automatically download fallback to Cleveland dataset instead of exiting
        download_uci_dataset("processed.cleveland.data")
        f = "processed.cleveland.data"

    if f.endswith(".data"):
        df = pd.read_csv(f, header=None, names=CLEVELAND_COLS, na_values="?")
    else:
        df = pd.read_csv(f)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={"thalch": "thalach"})

    tcol = next(c for c in ("num", "target", "condition", "output", "heartdisease", "hd")
                if c in df.columns)
    df = df.drop(columns=[c for c in ("id", "dataset") if c in df.columns])
    y = (pd.to_numeric(df[tcol]) > 0).astype(int).values      # 0 = absence, 1 = presence
    X = df.drop(columns=[tcol]).copy()

    for c in X.columns:                                       # missing-value handling
        if (not pd.api.types.is_numeric_dtype(X[c])) or pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(str)                           # categorical -> text (NaN -> 'nan')
        else:
            X[c] = X[c].fillna(X[c].median())                 # numeric -> median
    X = pd.get_dummies(X, drop_first=True, dtype=float)       # one-hot encode categoricals
    return X.values.astype(float), y, list(X.columns), f


def stratified_split(y, test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    te = []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        te.extend(idx[: int(round(len(idx) * test_frac))])
    te = np.array(sorted(te))
    tr = np.setdiff1d(np.arange(len(y)), te)
    return tr, te


class Standardizer:
    """z-score normalisation fitted on the training set only (no leakage)."""
    def fit(self, X):
        self.mu = X.mean(axis=0)
        self.sd = X.std(axis=0)
        self.sd[self.sd == 0] = 1.0
        return self

    def transform(self, X):
        return (X - self.mu) / self.sd


# ----------------------------------------------------------------------------------------
# MADALINE
# ----------------------------------------------------------------------------------------
def bsign(a):
    return np.where(a >= 0, 1, -1)


class Madaline:
    def __init__(self, n_inputs, n_hidden=5, lr=0.01, eta=0.5, seed=0):
        self.n_in, self.n_hidden = n_inputs, n_hidden
        self.lr, self.eta, self.seed = lr, eta, seed
        rng = np.random.default_rng(seed)
        self.W = rng.uniform(-0.1, 0.1, size=(n_hidden, n_inputs + 1))   # last column = bias
        self.history = []

    # ---- forward pass ----
    @staticmethod
    def _aug(X):
        X = np.atleast_2d(X)
        return np.hstack([X, np.ones((X.shape[0], 1))])

    def nets(self, X):
        """Net input of every hidden ADALINE, shape (n_samples, n_hidden)."""
        return self._aug(X) @ self.W.T

    def unit_outputs(self, X):
        return bsign(self.nets(X))                       # bipolar outputs +1 / -1

    def vote_sum(self, X):
        return self.unit_outputs(X).sum(axis=1)

    def decide(self, nets):
        """Majority-vote output unit. Ties (even sizes) are broken by the summed net input."""
        s = bsign(nets).sum(axis=1)
        tie = s == 0
        out = np.where(s > 0, 1, -1)
        out[tie] = bsign(nets[tie].sum(axis=1))
        return out

    def predict(self, X):
        return (self.decide(self.nets(X)) == 1).astype(int)     # back to {0,1}

    # ---- training ----
    def _pretrain(self, X, t, epochs):
        """Widrow-Hoff (LMS) pre-training of each ADALINE on its own bootstrap sample."""
        rng = np.random.default_rng(self.seed + 1)
        Xa = self._aug(X)
        for u in range(self.n_hidden):
            boot = rng.integers(0, len(X), len(X))
            for _ in range(epochs):
                for i in rng.permutation(boot):
                    err = t[i] - Xa[i] @ self.W[u]
                    self.W[u] += self.lr * err * Xa[i]

    def mri_step(self, xa, t, mode="minimal"):
        """
        One MADALINE Rule I update for a misclassified sample (xa is augmented, t is +1/-1).
        mode='minimal': try wrong units in order of smallest |net| (minimal disturbance) and
                        stop as soon as the majority vote becomes correct.
        mode='all'    : update every wrong unit at once (for comparison).
        Returns the list of updated unit indices.
        """
        nets = self.W @ xa
        outs = bsign(nets)
        wrong = [u for u in np.argsort(np.abs(nets)) if outs[u] != t]
        updated = []
        for u in wrong:
            net_u = self.W[u] @ xa
            # move this unit's net input a fraction eta of the way towards the target
            self.W[u] += self.eta * (t - net_u) * xa / (xa @ xa)
            updated.append(int(u))
            if mode == "minimal":
                if self.decide((self.W @ xa)[None, :])[0] == t:
                    break
        return updated

    def fit(self, X, y, epochs=60, pre_epochs=10, verbose=True):
        t = 2 * y - 1                                       # bipolar targets
        Xa = self._aug(X)
        rng = np.random.default_rng(self.seed + 2)
        self._pretrain(X, t, pre_epochs)
        self.history = []
        for ep in range(1, epochs + 1):
            n_err = 0
            for i in rng.permutation(len(X)):
                if self.decide((self.W @ Xa[i])[None, :])[0] != t[i]:
                    n_err += 1
                    self.mri_step(Xa[i], t[i], "minimal")
            self.history.append(n_err)
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"   epoch {ep:3d} | training errors during epoch: {n_err}")
            if n_err == 0:
                break
        return self

    def copy(self):
        m = Madaline(self.n_in, self.n_hidden, self.lr, self.eta, self.seed)
        m.W = self.W.copy()
        return m


# ----------------------------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------------------------
def confusion(y, p):
    tn = int(((y == 0) & (p == 0)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tp = int(((y == 1) & (p == 1)).sum())
    return tn, fp, fn, tp


def metrics(y, p):
    tn, fp, fn, tp = confusion(y, p)
    acc = (tp + tn) / max(len(y), 1)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(acc=acc, prec=prec, rec=rec, spec=spec, f1=f1, tn=tn, fp=fp, fn=fn, tp=tp)


LABEL = {0: "No disease", 1: "Disease"}


# ----------------------------------------------------------------------------------------
# Application state + helpers
# ----------------------------------------------------------------------------------------
class App:
    pass


app = App()


def ask(prompt, default, cast=int):
    s = input(f"{prompt} [default {default}]: ").strip()
    if not s:
        return default
    try:
        return cast(s)
    except ValueError:
        print("   invalid input, using default.")
        return default


def setup(path=None):
    X, y, names, f = load_heart(path)
    tr, te = stratified_split(y)
    sc = Standardizer().fit(X[tr])
    app.Xtr, app.Xte = sc.transform(X[tr]), sc.transform(X[te])
    app.ytr, app.yte = y[tr], y[te]
    app.sc, app.names, app.raw_te = sc, names, X[te]
    app.n_features = X.shape[1]
    app.is_binary = np.array([len(np.unique(X[:, j])) == 2 for j in range(X.shape[1])])
    app.model = None
    print(f"Loaded '{f}': {len(y)} records, {X.shape[1]} features after encoding "
          f"(train {len(tr)}, test {len(te)}).")
    print(f"Class balance: {int((y == 0).sum())} without disease, {int((y == 1).sum())} with disease.")


def train_model(n_hidden=5, lr=0.01, epochs=60, seed=0, verbose=True):
    m = Madaline(app.n_features, n_hidden=n_hidden, lr=lr, seed=seed)
    m.fit(app.Xtr, app.ytr, epochs=epochs, verbose=verbose)
    return m


def need_model():
    if app.model is None:
        print("   (no trained network yet - training one with default settings)")
        app.model = train_model()
    return app.model


def pick_patient(prompt="Test-set patient index"):
    n = len(app.yte)
    i = ask(f"{prompt} (0-{n - 1})", 0)
    if not 0 <= i < n:
        print("   out of range, using 0.")
        i = 0
    return i


# ----------------------------------------------------------------------------------------
# Menu options
# ----------------------------------------------------------------------------------------
def opt1_classify():
    print("\n=== 1. Heart Disease Classification using MADALINE ===")
    nh = ask("Number of hidden ADALINE units (odd number recommended)", 5)
    lr = ask("ADALINE learning rate (LMS pre-training)", 0.01, float)
    ep = ask("MRI training epochs", 60)
    print("Training...")
    t0 = time.time()
    app.model = train_model(nh, lr, ep)
    print(f"Done in {time.time() - t0:.2f}s")
    m = app.model
    mt, me = metrics(app.ytr, m.predict(app.Xtr)), metrics(app.yte, m.predict(app.Xte))
    print(f"\nTraining accuracy: {mt['acc'] * 100:.2f}%   Test accuracy: {me['acc'] * 100:.2f}%")
    print("\nFirst 10 test patients:")
    print(f"{'idx':>4} {'actual':>11} {'predicted':>11} {'votes(+1)':>10}  result")
    pred = m.predict(app.Xte)
    votes = (m.unit_outputs(app.Xte) == 1).sum(axis=1)
    for i in range(min(10, len(pred))):
        ok = "correct" if pred[i] == app.yte[i] else "WRONG"
        print(f"{i:>4} {LABEL[app.yte[i]]:>11} {LABEL[pred[i]]:>11} {votes[i]:>7}/{m.n_hidden}  {ok}")


def opt2_voting():
    print("\n=== 2. ADALINE Unit Voting Analyzer ===")
    m = need_model()
    i = pick_patient()
    x = app.Xte[i]
    nets = m.nets(x)[0]
    outs = bsign(nets)
    final = m.decide(nets[None, :])[0]
    print(f"\nPatient {i}: actual = {LABEL[app.yte[i]]}")
    print(f"{'unit':>5} {'net input':>11} {'output':>7}  vote")
    for u in range(m.n_hidden):
        print(f"{u + 1:>5} {nets[u]:>11.4f} {outs[u]:>+7d}  {'Disease' if outs[u] == 1 else 'No disease'}")
    pos, neg = int((outs == 1).sum()), int((outs == -1).sum())
    print(f"\nTally -> Disease: {pos}, No disease: {neg}")
    print(f"Majority decision: {LABEL[int(final == 1)]}  "
          f"({'correct' if int(final == 1) == app.yte[i] else 'WRONG'})")
    print(f"Winning margin: {abs(pos - neg)} vote(s) out of {m.n_hidden}")


def opt3_min_disturbance():
    print("\n=== 3. Minimal Disturbance Weight Update Challenge ===")
    m = need_model()
    Xall = np.vstack([app.Xtr, app.Xte])
    yall = np.concatenate([app.ytr, app.yte])
    wrong = np.where(m.predict(Xall) != yall)[0]
    if len(wrong) == 0:
        print("   The network classifies every record correctly - nothing to correct.")
        return
    k = ask(f"Which misclassified record to repair? (0-{len(wrong) - 1}, {len(wrong)} available)", 0)
    k = k if 0 <= k < len(wrong) else 0
    idx = wrong[k]
    x, tgt = Xall[idx], 2 * yall[idx] - 1
    xa = np.append(x, 1.0)
    print(f"\nRecord {idx}: actual = {LABEL[yall[idx]]}, network says {LABEL[1 - yall[idx]]}")
    print("Hidden unit nets/outputs before update:")
    nets = m.W @ xa
    for u in range(m.n_hidden):
        flag = "  <- wrong" if bsign(nets[u]) != tgt else ""
        print(f"   unit {u + 1}: net = {nets[u]:+.4f} output = {int(bsign(nets[u])):+d}{flag}")

    print(f"\n{'strategy':<20}{'units updated':<18}{'weight change (L2)':<22}{'fixed?':<8}{'train acc after'}")
    for mode, name in (("minimal", "Minimal disturbance"), ("all", "Update all wrong")):
        c = m.copy()
        before = c.W.copy()
        upd = c.mri_step(xa, tgt, mode)
        fixed = c.decide((c.W @ xa)[None, :])[0] == tgt
        change = np.linalg.norm(c.W - before)
        acc = metrics(app.ytr, c.predict(app.Xtr))["acc"] * 100
        print(f"{name:<20}{str([u + 1 for u in upd]):<18}{change:<22.4f}{str(bool(fixed)):<8}{acc:.2f}%")
    base = metrics(app.ytr, m.predict(app.Xtr))["acc"] * 100
    print(f"(training accuracy before any update: {base:.2f}%)")
    print("Minimal disturbance changes the fewest / least-confident units, so it normally damages "
          "what the network has already learned the least.")


def opt4_disagreement():
    print("\n=== 4. Hidden ADALINE Unit Disagreement Analyzer ===")
    m = need_model()
    outs = m.unit_outputs(app.Xte)
    yb = 2 * app.yte - 1
    pos = (outs == 1).sum(axis=1)
    unanimous = (pos == 0) | (pos == m.n_hidden)
    print(f"Unanimous patients: {unanimous.sum()} / {len(unanimous)} "
          f"({unanimous.mean() * 100:.1f}%),  split decisions: {(~unanimous).sum()}")
    pred = m.predict(app.Xte)
    if unanimous.any():
        print(f"Accuracy when units agree : {(pred[unanimous] == app.yte[unanimous]).mean() * 100:.2f}%")
    if (~unanimous).any():
        print(f"Accuracy when units split : {(pred[~unanimous] == app.yte[~unanimous]).mean() * 100:.2f}%")
    print("\nPer-unit test accuracy:")
    for u in range(m.n_hidden):
        print(f"   unit {u + 1}: {(outs[:, u] == yb).mean() * 100:.2f}%")
    print("\nPairwise agreement between units (%):")
    print("        " + "".join(f"U{v + 1:<6}" for v in range(m.n_hidden)))
    for a in range(m.n_hidden):
        row = "".join(f"{(outs[:, a] == outs[:, b]).mean() * 100:<7.1f}" for b in range(m.n_hidden))
        print(f"   U{a + 1:<4}{row}")
    margin = np.abs(2 * pos - m.n_hidden)
    order = np.argsort(margin)[:10]
    print("\nMost divided patients (smallest vote margin):")
    print(f"{'idx':>4} {'+1 votes':>9} {'-1 votes':>9} {'actual':>11} {'predicted':>11}")
    for i in order:
        print(f"{i:>4} {pos[i]:>9} {m.n_hidden - pos[i]:>9} {LABEL[app.yte[i]]:>11} {LABEL[pred[i]]:>11}")


def opt5_flip():
    print("\n=== 5. Patient Decision Flip Analyzer ===")
    m = need_model()
    i = pick_patient()
    k = ask("Perturbation size for numeric features (in standard deviations)", 1.0, float)
    x = app.Xte[i].copy()
    base_pred = int(m.predict(x)[0])
    base_votes = int(m.vote_sum(x)[0])
    print(f"\nPatient {i}: actual = {LABEL[app.yte[i]]}, predicted = {LABEL[base_pred]} "
          f"(vote sum {base_votes:+d})")
    print(f"\n{'feature':<22}{'change tested':<22}{'new decision':<14}{'vote sum':<10}flip?  min |shift| to flip")
    flips = []
    steps = np.arange(0.1, 3.01, 0.1)
    for j, name in enumerate(app.names):
        if app.is_binary[j]:
            vals = np.unique(app.Xtr[:, j])
            alt = vals[0] if abs(x[j] - vals[1]) < abs(x[j] - vals[0]) else vals[1]
            trials = [("toggle value", alt)]
            min_shift = "n/a (binary)"
        else:
            trials = [(f"+{k:g} sd", x[j] + k), (f"-{k:g} sd", x[j] - k)]
            min_shift = ">3 sd"
            for s in steps:
                if any(int(m.predict(np.where(np.arange(len(x)) == j, x[j] + sg * s, x))[0]) != base_pred
                       for sg in (1, -1)):
                    min_shift = f"{s:.1f} sd"
                    break
        for label, newv in trials:
            xp = x.copy()
            xp[j] = newv
            p = int(m.predict(xp)[0])
            flipped = p != base_pred
            if flipped:
                flips.append(name)
            print(f"{name:<22}{label:<22}{LABEL[p]:<14}{int(m.vote_sum(xp)[0]):<+10d}{'YES' if flipped else 'no':<7}{min_shift}")
    uniq = sorted(set(flips))
    print(f"\nFeatures that can flip this patient's decision at the tested perturbation: "
          f"{', '.join(uniq) if uniq else 'none'}")


def opt6_size():
    print("\n=== 6. MADALINE Network Size Challenge ===")
    sizes = [1, 3, 5, 7, 9, 11, 15]
    seeds = ask("Random seeds per size", 5)
    print(f"{'hidden units':>13} {'train acc %':>14} {'test acc %':>14} {'time (s)':>10}")
    best = (None, -1)
    for n in sizes:
        tr_a, te_a, t0 = [], [], time.time()
        for s in range(seeds):
            mdl = train_model(n, seed=s, verbose=False)
            tr_a.append(metrics(app.ytr, mdl.predict(app.Xtr))["acc"] * 100)
            te_a.append(metrics(app.yte, mdl.predict(app.Xte))["acc"] * 100)
        dt = (time.time() - t0) / seeds
        print(f"{n:>13} {np.mean(tr_a):>8.2f}+/-{np.std(tr_a):<4.2f} {np.mean(te_a):>8.2f}+/-{np.std(te_a):<4.2f} {dt:>10.2f}")
        if np.mean(te_a) > best[1]:
            best = (n, np.mean(te_a))
    print(f"\nBest mean test accuracy: {best[1]:.2f}% with {best[0]} hidden ADALINE units.")
    print("More units add voting diversity, but returns diminish and training time grows.")


def opt7_ambiguous():
    print("\n=== 7. Ambiguous Patient Record Detector ===")
    m = need_model()
    thr = ask("Ambiguity threshold on vote margin |votes(+1) - votes(-1)| / units", 0.34, float)
    s = m.vote_sum(app.Xte)
    margin = np.abs(s) / m.n_hidden
    mean_net = m.nets(app.Xte).mean(axis=1)
    amb = (margin <= thr) | (np.abs(mean_net) < 0.1)
    pred = m.predict(app.Xte)
    print(f"\nAmbiguous patients: {amb.sum()} of {len(amb)} test records")
    print(f"{'idx':>4} {'vote margin':>12} {'mean net':>10} {'actual':>11} {'predicted':>11}")
    for i in np.where(amb)[0][:20]:
        print(f"{i:>4} {margin[i]:>12.2f} {mean_net[i]:>+10.3f} {LABEL[app.yte[i]]:>11} {LABEL[pred[i]]:>11}")
    if amb.sum() > 20:
        print(f"   ... {amb.sum() - 20} more")
    if amb.any():
        print(f"\nAccuracy on ambiguous records     : {(pred[amb] == app.yte[amb]).mean() * 100:.2f}%")
    if (~amb).any():
        print(f"Accuracy on non-ambiguous records : {(pred[~amb] == app.yte[~amb]).mean() * 100:.2f}%")
    print("Ambiguous records are the ones a clinician should review manually.")


def opt8_report():
    print("\n=== 8. MADALINE Classification Performance Report ===")
    m = need_model()
    for name, X, y in (("TRAIN", app.Xtr, app.ytr), ("TEST", app.Xte, app.yte)):
        r = metrics(y, m.predict(X))
        print(f"\n--- {name} set ({len(y)} records) ---")
        print(f"Accuracy {r['acc'] * 100:.2f}% | Precision {r['prec'] * 100:.2f}% | "
              f"Recall (sensitivity) {r['rec'] * 100:.2f}% | Specificity {r['spec'] * 100:.2f}% | F1 {r['f1'] * 100:.2f}%")
        print("Confusion matrix:")
        print("                    Pred 0   Pred 1")
        print(f"   Actual 0 (none) {r['tn']:>8} {r['fp']:>8}")
        print(f"   Actual 1 (dis.) {r['fn']:>8} {r['tp']:>8}")
    yb = 2 * app.yte - 1
    outs = m.unit_outputs(app.Xte)
    print("\nPer-unit test accuracy: " +
          ", ".join(f"U{u + 1}={(outs[:, u] == yb).mean() * 100:.1f}%" for u in range(m.n_hidden)))
    print(f"Network: {m.n_hidden} ADALINE units, {app.n_features} inputs, majority-vote output, "
          f"LMS lr = {m.lr}, MRI eta = {m.eta}")
    print(f"MRI epochs used: {len(m.history)} (errors in last epoch: {m.history[-1] if m.history else 'n/a'})")


MENU = """
================ MADALINE Heart Disease System ================
 1. Heart Disease Classification using MADALINE
 2. ADALINE Unit Voting Analyzer
 3. Minimal Disturbance Weight Update Challenge
 4. Hidden ADALINE Unit Disagreement Analyzer
 5. Patient Decision Flip Analyzer
 6. MADALINE Network Size Challenge
 7. Ambiguous Patient Record Detector
 8. MADALINE Classification Performance Report
 9. Exit
==============================================================="""

ACTIONS = {"1": opt1_classify, "2": opt2_voting, "3": opt3_min_disturbance,
           "4": opt4_disagreement, "5": opt5_flip, "6": opt6_size,
           "7": opt7_ambiguous, "8": opt8_report}


def main():
    setup(sys.argv[1] if len(sys.argv) > 1 else None)
    while True:
        print(MENU)
        try:
            choice = input("Enter your choice (1-9): ").strip()
        except EOFError:
            break
        if choice == "9":
            print("Goodbye!")
            break
        action = ACTIONS.get(choice)
        if action is None:
            print("Invalid choice, please enter a number from 1 to 9.")
            continue
        action()


if __name__ == "__main__":
    main()


Dataset not found locally. Downloading fallback dataset from UCI: https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data ...
Download complete.
Loaded 'processed.cleveland.data': 303 records, 13 features after encoding (train 242, test 61).
Class balance: 164 without disease, 139 with disease.

================ MADALINE Heart Disease System ================
 1. Heart Disease Classification using MADALINE
 2. ADALINE Unit Voting Analyzer
 3. Minimal Disturbance Weight Update Challenge
 4. Hidden ADALINE Unit Disagreement Analyzer
 5. Patient Decision Flip Analyzer
 6. MADALINE Network Size Challenge
 7. Ambiguous Patient Record Detector
 8. MADALINE Classification Performance Report
 9. Exit
Enter your choice (1-9): 1

=== 1. Heart Disease Classification using MADALINE ===
Number of hidden ADALINE units (odd number recommended) [default 5]: 4
ADALINE learning rate (LMS pre-training) [default 0.01]: 0.1
MRI training epochs [default 60]: 52
Traini